In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, hashlib, subprocess, glob
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# Cell 2 - Arm B helpers (preregistration Amendment 2 B2.1).
# IPW reweights the TARGET evaluation set to the SOURCE class prior, then asks
# whether SHC undercoverage persists. Two facts we will demonstrate:
#   (a) class-conditional (Mondrian) coverage is INVARIANT to these weights by
#       construction (all points of a class share one weight), so the focal gap
#       CANNOT be a prior-shift artifact;
#   (b) marginal coverage MAY change; how much SHC marginal undercoverage
#       remains after reweighting is the prior-shift-vs-genuine decomposition.
# ESS guards reliability: focal/marginal with ESS<30 = inconclusive, not null.
# Randomized APS + matched draws, matching the coverage of record (nb13/nb17).
# =============================================================================
ALPHA=config.ALPHA_PRIMARY
R=getattr(config,'N_MATCHED_DRAWS',10)
def dseed(*p): return int(hashlib.sha256('|'.join(map(str,p)).encode()).hexdigest(),16)%(2**32)
def aps_scores_all(P, rng):
    order=np.argsort(-P,axis=1); sp=np.take_along_axis(P,order,1); cum=np.cumsum(sp,1)
    U=rng.random(len(P))[:,None]; ss=cum-(1-U)*sp
    out=np.empty_like(P); np.put_along_axis(out,order,ss,1); return out
def qhat(s,a):
    n=len(s); return np.inf if n<1 else float(np.quantile(s,min(np.ceil((n+1)*(1-a))/n,1.0),method='higher'))
def cov_indicators(eval_scores,y_eval,cal_all,y_cal,alpha,ncls):
    tc=cal_all[np.arange(len(y_cal)),y_cal]
    q={c:qhat(tc[y_cal==c],alpha) for c in range(ncls)}
    qv=np.array([q[c] for c in range(ncls)])
    cov=(eval_scores[np.arange(len(y_eval)),y_eval] <= qv[y_eval]).astype(float)
    return cov
def ipw_weights(y_src,y_eval,ncls):
    ps=np.array([np.mean(y_src==c) for c in range(ncls)])
    pe=np.array([np.mean(y_eval==c) for c in range(ncls)])
    w=np.divide(ps,pe,out=np.zeros_like(ps),where=pe>0)   # source/target prior ratio
    wi=w[y_eval]
    ess=float(wi.sum()**2/np.square(wi).sum()) if np.square(wi).sum()>0 else 0.0
    return w,wi,ess
print('Arm B helpers ready; alpha =',ALPHA,'| draws =',R)


Arm B helpers ready; alpha = 0.05 | draws = 10


In [3]:
# =============================================================================
# Cell 3 - CIC Arm B. Per realization, matched draws: SHC calibrates on the
# source pool, evaluated on D_eval; reweight D_eval to the source prior. Report
# unweighted vs IPW marginal coverage, focal (DoS) class-conditional invariance,
# and ESS. Focal DoS. CIC has real prior shift (S_lab ~0.5), so this is the
# meaningful test.
# =============================================================================
cic=pd.read_parquet(config.INTERIM_DIR/'cicids2017_primary.parquet')
wed=cic[cic['day']=='wednesday'].reset_index(drop=True); wed=wed[wed['label'].isin(['DoS','Benign'])].reset_index(drop=True)
CICC=['Benign','DoS']; CIC_NC=2; CIC_FOCAL=1; CIC_PROBS=config.DATA_DIR/'cic_probs'
REAL=['R1_holdout_Slowhttptest','R2_holdout_Slowloris','R3_holdout_GoldenEye',
      'R4_holdout_Slowloris_Slowhttptest','R5_holdout_GoldenEye_Slowloris']
def lab_cic(idx): return (wed.loc[idx,'label'].to_numpy()=='DoS').astype(int)
rows=[]
for name in REAL:
    spx=np.load(config.PROC_DIR/f'cic_{name}_srcpool_idx.npy'); tgx=np.load(config.PROC_DIR/f'cic_{name}_target_idx.npy')
    ysp,ytg=lab_cic(spx),lab_cic(tgx); mm=min(len(spx),len(tgx)//2)
    for f in sorted(CIC_PROBS.glob(f'{name}__*.npz')):
        _,arch,sp=f.stem.split('__'); seed=int(sp.replace('seed',''))
        d=np.load(f); Psp,Ptg=d['srcpool'],d['target']
        for draw in range(R):
            rng=np.random.default_rng(dseed(name,seed,arch,draw))
            tp=rng.permutation(len(ytg)); de=tp[:mm]; sc=rng.permutation(len(ysp))[:mm]
            P_de,yde=Ptg[de],ytg[de]; P_sc,ysc=Psp[sc],ysp[sc]
            es=aps_scores_all(P_de,np.random.default_rng(dseed(name,seed,arch,draw,'e')))
            cs=aps_scores_all(P_sc,np.random.default_rng(dseed(name,seed,arch,draw,'sc')))
            cov=cov_indicators(es,yde,cs,ysc,ALPHA,CIC_NC)
            w,wi,ess=ipw_weights(ysc,yde,CIC_NC)
            marg_unw=float(cov.mean()); marg_ipw=float((wi*cov).sum()/wi.sum())
            fm=yde==CIC_FOCAL
            foc_unw=float(cov[fm].mean()) if fm.any() else np.nan
            foc_ipw=float((wi[fm]*cov[fm]).sum()/wi[fm].sum()) if fm.any() and wi[fm].sum()>0 else np.nan
            row={'dataset':'cicids2017','realization':name,'arch':arch,'seed':seed,'draw':draw,
                 'ess':ess,'marg_unw':marg_unw,'marg_ipw':marg_ipw,'focal_unw':foc_unw,'focal_ipw':foc_ipw}
            for c in range(CIC_NC): row[f'w_{CICC[c]}']=float(w[c])
            rows.append(row)
cic_arm=pd.DataFrame(rows)
print('CIC Arm B rows:',len(cic_arm))
print(cic_arm.groupby('realization')[['ess','marg_unw','marg_ipw','focal_unw','focal_ipw']].mean().round(4).to_string())


CIC Arm B rows: 1500
                                          ess  marg_unw  marg_ipw  focal_unw  focal_ipw
realization                                                                            
R1_holdout_Slowhttptest             1951.2476    0.9490    0.8999     0.8548     0.8548
R2_holdout_Slowloris                4394.3648    0.9383    0.7135     0.4978     0.4978
R3_holdout_GoldenEye                8033.7705    0.9402    0.8411     0.7395     0.7395
R4_holdout_Slowloris_Slowhttptest   6203.6285    0.9263    0.6078     0.2923     0.2923
R5_holdout_GoldenEye_Slowloris     11793.4555    0.9281    0.7891     0.6352     0.6352


In [ ]:
# =============================================================================
# Cell 4 - UGR Arm B. UGR has NO prior shift (S_lab=0, balanced support), so
# weights are near-uniform and ESS near n: this is the null check confirming the
# UGR failures are not prior shift either. Focal nerisbotnet.
# =============================================================================
UGR=config.DATASETS_DIR/'ugr16'
usrc=pd.read_parquet(UGR/'july_week5.parquet'); utgt=pd.read_parquet(UGR/'august_week1.parquet')
for dd in (usrc,utgt): dd['label']=dd['label'].astype(str).str.strip().str.lower()
UK=['background','dos','scan11','scan44','nerisbotnet']
usrc=usrc[usrc.label.isin(UK)].reset_index(drop=True); utgt=utgt[utgt.label.isin(UK)].reset_index(drop=True)
UCL=sorted(UK); U2I={c:i for i,c in enumerate(UCL)}; UGR_NC=len(UCL); UGR_FOCAL=U2I['nerisbotnet']
def strat(df,fr,seed,col='label'):
    rng=np.random.default_rng(seed); nm=list(fr); f=np.array([fr[k] for k in nm],float); big=nm[int(np.argmax(f))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rng.shuffle(idx); n=len(idx)
        c=np.floor(f*n).astype(int); c[nm.index(big)]+=n-c.sum(); kk=0
        for a2,q in zip(nm,c): a.loc[idx[kk:kk+q]]=a2; kk+=q
    return a
usrc=usrc.assign(partition=strat(usrc,config.SPLIT_FRACTIONS,20260725).values)
y_sp=usrc[usrc.partition=='source_cal_pool']['label'].map(U2I).to_numpy(); y_tg=utgt['label'].map(U2I).to_numpy()
UGR_PROBS=config.DATA_DIR/'ugr16_probs'; mmU=min(len(y_sp),len(y_tg)//2)
rows=[]
for f in sorted(UGR_PROBS.glob('ugr16__*.npz')):
    _,arch,sp=f.stem.split('__'); seed=int(sp.replace('seed',''))
    d=np.load(f); Psp,Ptg=d['srcpool'],d['target']
    for draw in range(R):
        rng=np.random.default_rng(dseed('ugr16',seed,arch,draw))
        tp=rng.permutation(len(y_tg)); de=tp[:mmU]; sc=rng.permutation(len(y_sp))[:mmU]
        P_de,yde=Ptg[de],y_tg[de]; P_sc,ysc=Psp[sc],y_sp[sc]
        es=aps_scores_all(P_de,np.random.default_rng(dseed('ugr16',seed,arch,draw,'e')))
        cs=aps_scores_all(P_sc,np.random.default_rng(dseed('ugr16',seed,arch,draw,'s')))
        cov=cov_indicators(es,yde,cs,ysc,ALPHA,UGR_NC)
        w,wi,ess=ipw_weights(ysc,yde,UGR_NC)
        marg_unw=float(cov.mean()); marg_ipw=float((wi*cov).sum()/wi.sum())
        fm=yde==UGR_FOCAL
        foc_unw=float(cov[fm].mean()) if fm.any() else np.nan
        foc_ipw=float((wi[fm]*cov[fm]).sum()/wi[fm].sum()) if fm.any() and wi[fm].sum()>0 else np.nan
        row={'dataset':'ugr16','realization':'july_to_august','arch':arch,'seed':seed,'draw':draw,
             'ess':ess,'marg_unw':marg_unw,'marg_ipw':marg_ipw,'focal_unw':foc_unw,'focal_ipw':foc_ipw}
        for c in range(UGR_NC): row[f'w_{UCL[c]}']=float(w[c])
        rows.append(row)
ugr_arm=pd.DataFrame(rows)
print('UGR Arm B rows:',len(ugr_arm))
print(ugr_arm[['ess','marg_unw','marg_ipw','focal_unw','focal_ipw']].mean().round(4).to_string())


In [ ]:
# =============================================================================
# Cell 5 - the Arm B verdict: does SHC undercoverage persist after IPW?
# Class-conditional (focal) is near-identical weighted vs unweighted (invariance,
# exact up to float since a class shares one weight). Marginal shows the
# prior-shift-vs-genuine split.
# =============================================================================
def summ(df,label):
    d=df.copy(); ess=d['ess'].mean(); ess_min=d['ess'].min()
    mu,mi=d['marg_unw'].mean(),d['marg_ipw'].mean(); fu,fi=d['focal_unw'].mean(),d['focal_ipw'].mean()
    print(f'\n[{label}]  mean ESS={ess:.0f} (min {ess_min:.0f}) of D_eval size')
    print(f'  MARGINAL  coverage: unweighted={mu:.4f}  IPW-reweighted={mi:.4f}  (shift {mi-mu:+.4f})')
    print(f'  FOCAL c-c coverage: unweighted={fu:.4f}  IPW-reweighted={fi:.4f}  (shift {fi-fu:+.4f} <- ~0 = invariant)')
    return {'dataset':label,'mean_ess':round(ess,1),'min_ess':round(ess_min,1),
            'marg_unweighted':round(mu,4),'marg_ipw':round(mi,4),'marg_shift':round(mi-mu,4),
            'focal_unweighted':round(fu,4),'focal_ipw':round(fi,4),'focal_shift':round(fi-fu,4),
            'focal_invariant':bool(abs(fi-fu)<0.02),'ess_inconclusive_flag':bool(ess_min<30)}
v_cic=summ(cic_arm,'cicids2017'); v_ugr=summ(ugr_arm,'ugr16')

verdict={'analysis':'Arm B IPW (Amendment 2 B2.1): reweight target to source prior; does SHC undercoverage persist',
   'cicids2017':v_cic,'ugr16':v_ugr,
   'reads':('focal_shift ~ 0 confirms class-conditional coverage is invariant to prior shift, so the focal '
            'gap is NOT a prior-shift artifact. marg_shift shows how much marginal undercoverage was prior '
            'shift (removed by IPW) vs genuine (persists). NSL Arm B already committed (armb_weights_nslkdd.csv).')}
(config.REPORTS_DIR/'armb_verdict.json').write_text(json.dumps(verdict,indent=2))

# per-realization weights + ESS, matching the NSL armb_weights format
wcols_cic=[f'w_{c}' for c in CICC]
cicw=cic_arm.groupby('realization')[['ess']+wcols_cic].mean().reset_index(); cicw.insert(0,'dataset','cicids2017')
cicw.to_csv(config.REPORTS_DIR/'armb_weights_cicids2017.csv',index=False)
wcols_ugr=[f'w_{c}' for c in UCL]
ugrw=ugr_arm.groupby('realization')[['ess']+wcols_ugr].mean().reset_index(); ugrw.insert(0,'dataset','ugr16')
ugrw.to_csv(config.REPORTS_DIR/'armb_weights_ugr16.csv',index=False)
cic_arm.to_csv(config.REPORTS_DIR/'armb_detail_cicids2017.csv',index=False)
ugr_arm.to_csv(config.REPORTS_DIR/'armb_detail_ugr16.csv',index=False)
print('\nCIC weights:'); print(cicw.round(4).to_string(index=False))
print('\nUGR weights:'); print(ugrw.round(4).to_string(index=False))
print('\n',json.dumps(verdict,indent=2))

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb23: Arm B IPW on CIC + UGR (focal invariance + marginal prior-shift decomposition)')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)
